# Maintenance Efficiency Analysis

Measures maintenance-team efficiency per property: reopen rates, resolution times, ticket
age, and ticket repetition, for buildings owned over the past year.

Stack: PySpark, Microsoft Fabric lakehouse.

Note: internal identifiers are anonymized, and outputs are cleared because the originals
ran on internal data.


In [ ]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql.functions import col, when
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Load and Wrangle Data

In [ ]:
# Load the processed data
tickets = spark.sql("SELECT * FROM Silver_Lakehouse.rp.maintenance_tickets_processed")


# create a new column 'reopen'
tickets = tickets.withColumn('reopen', when(col('reopen_count') > 0, True).otherwise(False))

## Active Buildings In The Last Year 
Filter for Buildings that we've owned for the past13 months (take out sold sites, and sites acquired in the past 13 months) 

In [ ]:
# # loading data
# hierarchy_spark = spark.sql("SELECT * FROM Silver_Lakehouse.dbo.DIM_RE_HIERARCHY")

# # Iterate over all columns and replace blank values with null values
# for column in hierarchy_spark.columns:
#     hierarchy_spark = hierarchy_spark.withColumn(column, when(col(column) == '', None).otherwise(col(column)))

# # # Exclude commercial and landlease
# # hierarchy_spark = hierarchy_spark.filter((col('Building_Class')!= "COMMERCIAL") & (col('Building_Class')!= "LANDLEASE"))

# # only usefull columns
# cols = ['Building', 'Building_Number', 'Site_Number', 'Site_Number-Name', 'Acquisition_Date', 'Country',
#         'Region',
#         'City', 
#         'House_Number',
#         'Street_Name',
#         'Postal_Code',
#         'Building_Name',
#         'Building_Class',
#         'Building_Type',
#         'Company',
#         'Company_Code',
#         'Profit_Centre_Name',
#         'Profit_Centre',
#         'Valid_from',
#         'Valid_to',
#         'Locations',
#         'Suite_Count',
#         'BPC_CC',
#         'Construction_Category']

# hierarchy_spark = hierarchy_spark.select(*cols)

# hierarchy_spark = hierarchy_spark.withColumnRenamed('Site_Number', 'site')

In [ ]:
# from pyspark.sql.functions import to_date, col, when, max as spark_max, date_sub, lit

# # extracting last day and one year ago date from tickets data
# last_day = tickets.select(spark_max('created_at')).collect()[0][0]
# one_year_ago = date_sub(lit(last_day), 366)

# # filter the hierarchy data
# hierarchy_filtered = hierarchy_spark.filter((col('Acquisition_Date') < one_year_ago) & (col('Valid_to') > last_day))

# # convert to pandas
# hierarchy_filtered = hierarchy_filtered.toPandas()

# # Site Properties
# site_properties = hierarchy_filtered.groupby('site').agg({'City': 'first',
#                                                      'Region': 'first', 
#                                                      'Site_Number-Name': 'first', 
#                                                      'Construction_Category': 'first', 
#                                                      'Suite_Count':'sum'}).reset_index()

In [ ]:
from pyspark.sql.functions import first, sum

# Group by 'site' and aggregate the required columns
properties = tickets.groupBy(['site','Building_Number']).agg(
    first('City').alias('City'),
    first('Region').alias('Region'),
    first('Site_Number-Name').alias('Site_Number-Name'),
    first('Construction_Category').alias('Construction_Category'),
    first('Suite_Count').alias('Suite_Count')
)

# Convert to a DataFrame and display the result
site_Properties = properties.groupBy('site').agg(
    first('City').alias('City'),
    first('Region').alias('Region'),
    first('Site_Number-Name').alias('Site_Number-Name'),
    first('Construction_Category').alias('Construction_Category'),
    sum('Suite_Count').alias('Suite_Count')
)

site_properties = site_Properties.toPandas()

## Last Year Data

In [ ]:
from pyspark.sql.functions import to_date, col, when, max as spark_max, date_sub, lit
# Filter the last year data
last_day = tickets.select(spark_max('created_at')).collect()[0][0]
one_year_ago = date_sub(lit(last_day), 366)

df_last_year_spark = tickets.filter(tickets['created_at'] >= one_year_ago)

df_last_year = df_last_year_spark.toPandas()

# Descriptive Table

## Reopen rate, Resolution Time

In [ ]:
# Calculate the reopen rate and avg resolution time for each property
descriptive_table = (df_last_year.groupby('site')
                        .aggregate(reopen_count=('reopen', 'sum'),
                                   reopen_rate=('reopen', 'mean'), 
                                   avg_resolution_time=('resolution_time', 'mean'),
                                   avg_response_time=('response_time', 'mean'), 
                                   avg_time_to_need_contractor=('contractor_need_time', 'mean'), 
                                   avg_time_to_facilitate_contractor=('contractor_finding_time', 'mean'), 
                                   ticket_count=('id', 'count')).reset_index())

# merge with properties
descriptive_table = descriptive_table.merge(site_properties, on='site', how='inner')

# calculate z-scores for reopen rate and avg resolution time
descriptive_table['reopen_reate_z_score'] = (descriptive_table['reopen_rate'] - descriptive_table['reopen_rate'].mean()) / descriptive_table['reopen_rate'].std()
descriptive_table['avg_resolution_time_z_score'] = (descriptive_table['avg_resolution_time'] - descriptive_table['avg_resolution_time'].mean()) / descriptive_table['avg_resolution_time'].std()
descriptive_table['avg_response_time_z_score'] = (descriptive_table['avg_response_time'] - descriptive_table['avg_response_time'].mean()) / descriptive_table['avg_response_time'].std()
descriptive_table['avg_time_to_need_contractor_z_score'] = (descriptive_table['avg_time_to_need_contractor'] - descriptive_table['avg_time_to_need_contractor'].mean()) / descriptive_table['avg_time_to_need_contractor'].std()
descriptive_table['avg_time_to_facilitate_contractor_z_score'] = (descriptive_table['avg_time_to_facilitate_contractor'] - descriptive_table['avg_time_to_facilitate_contractor'].mean()) / descriptive_table['avg_time_to_facilitate_contractor'].std()

# Ticket per suit
descriptive_table['ticket_per_suit'] = descriptive_table['ticket_count'] / descriptive_table['Suite_Count']
descriptive_table['ticket_per_suit_z_score'] = (descriptive_table['ticket_per_suit'] - descriptive_table['ticket_per_suit'].mean()) / descriptive_table['ticket_per_suit'].std()

## Ticket Age

In [ ]:
# Ticket Age
df = df_last_year[~((df_last_year.status.isin(['cancelled', 'cancelled_dup', 'Closed']))&(df_last_year.completion_date.isna()))].copy()

# set NaN completion date to today
max_date = pd.Timestamp(df['completion_date'].max().date()) # Convert the maximum date to a Timestamp
df['completion_date'] = df['completion_date'].fillna(max_date)

df['ticket_age'] = (df['completion_date'] - df['created_at']).dt.days
df['ticket_age'] = df['ticket_age'].apply(lambda x:np.maximum(0, x))

ticket_age = (df.groupby('site')
              .aggregate(ticket_counts=('id', 'count'), 
                         avg_ticket_age=('ticket_age', 'mean'))
                        .reset_index())
ticket_age['avg_ticket_age_z_score'] = (ticket_age['avg_ticket_age'] - ticket_age['avg_ticket_age'].mean()) / ticket_age['avg_ticket_age'].std()

descriptive_table = descriptive_table.merge(ticket_age, on='site', how='left')

## Ticket Repetition

In [ ]:

# Counting repeatitons by considering all the related tickets
# Filter out cancelled_dup and closed tickets with no completion date
df = df_last_year[~((df_last_year.status.isin(['cancelled_dup', 'Closed'])) & df_last_year.completion_date.isna())].copy()

# Fill missing completion_date with updated_at where the status is 'cancelled'
mask = (df['status'] == 'cancelled') & df['completion_date'].isna()
df.loc[mask, 'completion_date'] = df.loc[mask, 'updated_at']

# Missing unit_ids
# Step 1: Separate common area tickets (where unit_id is missing)
common_area_df = df[df['unit_id'].isna()].copy()

# Step 2: Handle common area tickets (e.g., mark all as not reopened)
common_area_df['is_reopened'] = False  # Assume no reopening for common areas

# Step 3: Handle regular tickets with valid unit_id
df = df.dropna(subset=['unit_id'])  # Drop common area tickets

# Sort the data by created_at to ensure correct comparison order
df = df.sort_values(by=['site', 'unit_id', 'category', 'suite_location', 'created_at'])

# Check initial shape
initial_shape = df.shape
print(f"Initial DataFrame Shape: {initial_shape}")

# Function to detect reopened tickets with the correct logic
def detect_reopen(group):
    group = group.reset_index(drop=True)  # Ensure index is sequential
    group['is_reopened'] = False  # Initialize column with False

    # Iterate over the group to compare each ticket against previous valid completions
    for i, row in group.iterrows():
        # Filter out the current row from valid previous completions
        valid_completions = group.iloc[:i].dropna(subset=['completion_date'])

        # Check if the current ticket is a reopened case
        if not valid_completions.empty:
            is_reopened = any(
                (row['created_at'] > valid_completions['completion_date']) &
                (row['created_at'] - valid_completions['completion_date']).dt.days.between(0, 30)
            )
            group.at[i, 'is_reopened'] = is_reopened

    return group

# Apply the function to each property group
df = df.groupby(['site', 'unit_id', 'category', 'suite_location']).apply(detect_reopen).reset_index(drop=True)

# Ensure that any ticket with 'Other' category is marked as not reopened
df.loc[df['category']=='Other', 'is_reopened'] = False

#  Combine both datasets
final_df = pd.concat([df, common_area_df], ignore_index=True)

# Check shape after reopening detection
after_reopen_shape = df.shape
print(f"Shape After Reopening Detection: {after_reopen_shape}") 

# Check if any rows were lost
if initial_shape[0] != after_reopen_shape[0]:
    print(f"Warning: {initial_shape[0] - after_reopen_shape[0]} rows were lost!")
else:
    print("No rows were lost during the operation.")

# Calculate site performance metrics
repeat_table = final_df.groupby('site').aggregate(
    repeat_count=('is_reopened', 'sum'),
    repeat_rate=('is_reopened', 'mean')
).sort_values('repeat_rate', ascending=False).reset_index()

# Calculate z-scores for repeat rate to flag anomalies
repeat_table['repeat_rate_z_score'] = (
    (repeat_table['repeat_rate'] - repeat_table['repeat_rate'].mean()) / repeat_table['repeat_rate'].std()
)

descriptive_table = descriptive_table.merge(repeat_table[['site', 'repeat_count', 'repeat_rate', 'repeat_rate_z_score']], on='site', how='left')

## Changing table structure

In [ ]:
from datetime import datetime

# Get the current date
current_date = datetime.now().date()

descriptive_table['date'] = current_date

# melting the table
melted_descriptive_table = descriptive_table.melt(id_vars=['site', 'date'],
                                                  value_vars=['ticket_count', 'ticket_per_suit','ticket_per_suit_z_score','reopen_count', 
                                                  'reopen_rate', 'reopen_reate_z_score',
                                                  'avg_resolution_time', 'avg_resolution_time_z_score', 
                                                  'avg_ticket_age', 'avg_ticket_age_z_score', 
                                                  'repeat_count', 'repeat_rate', 'repeat_rate_z_score', 
                                                  'avg_response_time', 'avg_response_time_z_score', 
                                                  'avg_time_to_need_contractor', 'avg_time_to_need_contractor_z_score', 
                                                  'avg_time_to_facilitate_contractor', 'avg_time_to_facilitate_contractor'],
                                                  var_name='attribute', 
                                                  value_name='value'
                                                  )

# Updating The Lakehouse

In [ ]:
from pyspark.sql.functions import current_date

# Create the schema if it does not exist
# spark.sql("CREATE SCHEMA IF NOT EXISTS rp")

descriptive_table_spark = spark.createDataFrame(melted_descriptive_table)

# Save the DataFrame as a table in the specified schema (append mode)
descriptive_table_spark.write.mode("append").format("delta").save("abfss://Gold_AnalyticalData@onelake.dfs.fabric.microsoft.com/Gold_Lakehouse.Lakehouse/Tables/rp/descriptive_stats")